# 02 — Quantization tour

Quantization trades memory (and sometimes a little recall) for much faster distance computations and much smaller on-disk footprints. This notebook measures the tradeoff for three quantizers against a `FLAT` baseline using a small synthetic dataset.

| Quantizer | Bits per component | Memory reduction | Typical recall loss |
|---|---|---|---|
| `NONE` (float32) | 32 | 1× | baseline |
| `SQ8` | 8 | 4× | <1% |
| `PQ` | 1 per sub-vector coordinate (at `M=16`, `nbits=8`) | 32–64× | 1–5% |
| `BQ` | 1 | 32× | 5–15% — pair with rescore |

In [ ]:
%jars /home/jovyan/work/vectors/vectors-core/build/libs/vectors-core-*.jar
%jars /home/jovyan/work/vectors/vectors-storage/build/libs/vectors-storage-*.jar
%jars /home/jovyan/work/vectors/vectors-quantization/build/libs/vectors-quantization-*.jar
%jars /home/jovyan/work/vectors/vectors-hnsw/build/libs/vectors-hnsw-*.jar
%jars /home/jovyan/work/vectors/vectors-db/build/libs/vectors-db-*.jar

In [ ]:
import com.integrallis.vectors.core.SimilarityFunction;
import com.integrallis.vectors.db.Document;
import com.integrallis.vectors.db.IndexType;
import com.integrallis.vectors.db.QuantizerKind;
import com.integrallis.vectors.db.SearchRequest;
import com.integrallis.vectors.db.SearchResult;
import com.integrallis.vectors.db.VectorCollection;
import java.util.*;

final int DIMENSION = 64;
final int N = 2_000;
final Random rnd = new Random(17L);

float[] unit(int dim) {
    float[] v = new float[dim];
    float norm = 0f;
    for (int i = 0; i < dim; i++) { v[i] = (float) rnd.nextGaussian(); norm += v[i]*v[i]; }
    norm = (float) Math.sqrt(norm);
    if (norm > 0f) for (int i = 0; i < dim; i++) v[i] /= norm;
    return v;
}

List<Document> corpus = new ArrayList<>(N);
for (int i = 0; i < N; i++) corpus.add(Document.of("doc-" + i, unit(DIMENSION), null));
List<float[]> queries = new ArrayList<>(20);
for (int i = 0; i < 20; i++) queries.add(unit(DIMENSION));
System.out.printf("corpus: %d x %d-dim, queries: %d%n", N, DIMENSION, queries.size());

In [ ]:
// Ground truth: FLAT, no quantization. Every query has a reference top-10.
VectorCollection gt = VectorCollection.builder()
    .dimension(DIMENSION).metric(SimilarityFunction.COSINE).indexType(IndexType.FLAT)
    .autoCommitThreshold(Integer.MAX_VALUE).build();
for (Document d : corpus) gt.add(d);
gt.commit();

List<Set<String>> truth = new ArrayList<>();
for (float[] q : queries) {
    SearchResult r = gt.search(SearchRequest.builder(q, 10).build());
    Set<String> ids = new HashSet<>();
    for (SearchResult.Hit h : r.hits()) ids.add(h.document().id());
    truth.add(ids);
}
gt.close();
System.out.println("ground truth computed");

In [ ]:
// Helper: build an HNSW collection with the requested quantizer, measure recall@10.
double recallAt10(QuantizerKind q) {
    VectorCollection col = VectorCollection.builder()
        .dimension(DIMENSION).metric(SimilarityFunction.COSINE).indexType(IndexType.HNSW)
        .quantizer(q).hnswM(16).hnswEfConstruction(100)
        .autoCommitThreshold(Integer.MAX_VALUE).build();
    for (Document d : corpus) col.add(d);
    col.commit();
    int matched = 0, total = 0;
    for (int i = 0; i < queries.size(); i++) {
        SearchResult r = col.search(SearchRequest.builder(queries.get(i), 10).build());
        for (SearchResult.Hit h : r.hits()) if (truth.get(i).contains(h.document().id())) matched++;
        total += 10;
    }
    col.close();
    return matched / (double) total;
}

for (QuantizerKind q : List.of(QuantizerKind.NONE, QuantizerKind.SQ8, QuantizerKind.PQ, QuantizerKind.BQ)) {
    try {
        double recall = recallAt10(q);
        System.out.printf("%-6s -> recall@10 = %.3f%n", q, recall);
    } catch (UnsupportedOperationException ex) {
        System.out.printf("%-6s -> unsupported in this HNSW build: %s%n", q, ex.getMessage());
    }
}

## Reading the results

- **`NONE`** is the reference — HNSW with full-precision float32 should land at or near 1.000.
- **`SQ8`** typically matches `NONE` to within 0.01.
- **`PQ`** trades a few recall points for ~32× memory reduction; combine with an `ivfRescoreFactor` on IVF_PQ for production.
- **`BQ`** (sign-bit binary) drops recall noticeably in this tiny dataset — it shines on very high-dimensional data (1024+) with a rescore pass.

For publication-grade numbers (latency and build throughput), run `./gradlew :vectors-bench:jmh`.